# 03 Ranking Evaluation

This notebook explains what happens after candidate generation.

The demo reranker is intentionally simple and transparent:
- dot product between user interests and campaign weights,
- plus bid,
- plus a small freshness term,
- minus a lightweight frequency penalty.

That makes the ranking behavior easy to inspect while still being realistic enough for offline evaluation.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.candidate import build_indexes, filter_campaigns_for_user, generate_candidates_in_memory
from app.models import Campaign, UserProfile
from app.ranking import rerank_campaigns, score_campaign
from data.common import click_probability, read_jsonl, truth_score
from experiments.evaluate import evaluate_synthetic

DATASET_DIR = REPO_ROOT / 'data' / 'generated' / 'synthetic'
users = [UserProfile.model_validate(row) for row in read_jsonl(DATASET_DIR / 'users.jsonl')]
campaigns = [Campaign.model_validate(row) for row in read_jsonl(DATASET_DIR / 'campaigns.jsonl')]
campaign_by_id = {campaign.campaign_id: campaign for campaign in campaigns}
indexes = build_indexes(campaigns)

sample_user = next(user for user in users if generate_candidates_in_memory(user, indexes, max_candidates=50, strong_signal_count=2))
sample_user

UserProfile(user_id='maid_00000', geo='CA', device='Android', age_bucket='35-44', interests={'camping': 0.3322, 'family': 0.1344, 'finance': 0.4761, 'fitness': 0.4114, 'foodie': 0.1935, 'gaming': 0.6023, 'home_improvement': 0.875, 'luxury': 0.1125, 'pet_care': 0.2847, 'streaming': 0.4398, 'tech': 0.4962, 'travel': 0.7217}, segments=['home_improvement_high', 'travel_high', 'gaming_medium'], identity_tokens=['id_00000_01', 'id_00000_02', 'id_00000_03'], state='ON', postal_code='M4B1B3', device_type='tablet', card_tier='Standard', spend_tier='low', frequency_history={'c00015': 2, 'c00141': 2, 'c00202': 1, 'c00378': 1, 'c00731': 2, 'c00908': 1, 'c00918': 1, 'c01193': 1, 'c01258': 1, 'c01371': 3, 'c01873': 2, 'c02484': 2}, impression_count=25)

## From Candidate IDs To Ranked Ads

The notebook below follows the same steps as the API:
1. generate candidate IDs,
2. materialize campaign objects,
3. enforce eligibility,
4. score and sort the survivors.

In [2]:
candidate_ids = generate_candidates_in_memory(sample_user, indexes, max_candidates=50, strong_signal_count=2)
candidate_campaigns = [campaign_by_id[campaign_id] for campaign_id in candidate_ids if campaign_id in campaign_by_id]
eligible_campaigns = filter_campaigns_for_user(sample_user, candidate_campaigns)
ranked = rerank_campaigns(sample_user, eligible_campaigns, top_k=10)

pd.Series(
    {
        'candidate_ids': len(candidate_ids),
        'eligible_campaigns_after_filter': len(eligible_campaigns),
        'top_k_returned': len(ranked),
    }
)

candidate_ids                      50
eligible_campaigns_after_filter     2
top_k_returned                      2
dtype: int64

## Score Components

The returned score is decomposed into named parts so the ranking can be explained to a teammate or customer.
That same component view is what the `/rank` endpoint returns in the demo.

In [3]:
score_frame = pd.DataFrame(
    [
        {
            'campaign_id': item.campaign_id,
            'score': item.score,
            **item.score_components,
            'click_probability': round(click_probability(sample_user, campaign_by_id[item.campaign_id]), 6),
            'truth_score': round(truth_score(sample_user, campaign_by_id[item.campaign_id]), 6),
        }
        for item in ranked
    ]
)
score_frame

,campaign_id,score,interest,bid,freshness,frequency_penalty,click_probability,truth_score
0,c00202,6.073303,1.764303,3.823,0.511000,0.025,0.905788,3.863258
1,c00522,5.935757,3.122816,2.813,0.024942,0.025,0.933616,4.243610


## Single-Campaign Explanation

For one campaign, it is useful to inspect both the raw campaign object and the scored breakdown.

In [4]:
best = ranked[0]
best_campaign = campaign_by_id[best.campaign_id]

display(pd.json_normalize(best_campaign.model_dump()))
display(pd.Series(score_campaign(sample_user, best_campaign).model_dump()))

,campaign_id,geo,device,required_segments,any_of_segments,none_of_segments,card_tiers,geo_states,geo_postal_codes,device_types,...,bid,freshness_boost,age_in_days,weights.camping,weights.family,weights.finance,weights.fitness,weights.pet_care,weights.streaming,weights.travel
0,c00202,[*],[Android],[],"[streaming_high, family_medium, camping_medium...",[],[*],[],[],"[tablet, mobile]",...,3.823,0.511,0,1.309,1.3355,-0.2353,0.7844,-0.2711,1.196,0.6796


campaign_id                                                    c00202
score                                                        6.073303
score_components    {'interest': 1.764303, 'bid': 3.823, 'freshnes...
dtype: object

## Offline Quality Metrics

The synthetic evaluator reports both ranking quality and candidate-generation recall.
That separation matters because a perfect reranker cannot recover a campaign that was dropped during coarse retrieval.

In [5]:
results = evaluate_synthetic(DATASET_DIR, top_k=5, sample_users=250)
pd.Series(results)

users_evaluated                        250
strategy                       union_probe
ndcg_at_k                            0.917
precision_at_k                      0.9901
recall_at_k                         0.2594
f1_at_k                             0.4007
candidate_generation_recall         0.4704
dtype: object